# U3-03 · Recomendación de restaurantes y aprendizaje por refuerzo

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 3*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con ▶). Con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) las redes se entrenan mucho más deprisa.

> Datos: Medellín, R. y Serna, J. (2012). *Restaurant & consumer data*. UCI, CC BY 4.0. https://doi.org/10.24432/C5DP41
> Entorno Taxi-v4 de Gymnasium (Towers et al., 2024), propuesto por Dietterich (2000).

In [ ]:
%pip install -q gymnasium
# Descarga los datos y módulos auxiliares (solo la primera vez)
import os, urllib.request
for f in ["rating_final.csv", "geoplaces2.csv", "chefmozcuisine.csv", "usercuisine.csv"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve("https://machinelearning-formacion.github.io/ifcd107/recursos/datos/u3/" + f, f)

## 3.5 · Sistemas de recomendación

`k1_valoraciones.py`

In [ ]:
import pandas as pd

r = pd.read_csv("rating_final.csv")
g = pd.read_csv("geoplaces2.csv", usecols=["placeID", "name",
                                           "city", "price"])
nu, nr = r.userID.nunique(), r.placeID.nunique()
print(f"{len(r)} valoraciones, {nu} usuarios, {nr} restaurantes")
print(f"matriz usuario x restaurante llena al "
      f"{len(r) / (nu * nr):.1%}")
print("valoraciones 0/1/2:", r.rating.value_counts().sort_index()
      .to_dict())
por_u = r.groupby("userID").size()
print(f"por usuario: mediana {por_u.median():.0f},"
      f" mínimo {por_u.min()}, máximo {por_u.max()}")
print("ciudades:", g.city.str.lower().str.strip().value_counts()
      .head(4).to_dict())
top = (r.groupby("placeID").rating.agg(["mean", "size"])
       .query("size >= 10").sort_values("mean", ascending=False)
       .head(3).join(g.set_index("placeID").name))
print(top.round(2).to_string())

`k2_referencias.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

r = pd.read_csv("rating_final.csv")
ent, pru = train_test_split(r, test_size=0.2, random_state=0)
rmse = lambda p: np.sqrt(np.mean((pru.rating.to_numpy() - p) ** 2))
mu = ent.rating.mean()
r0 = rmse(np.full(len(pru), mu))
print(f"media global {mu:.3f}: RMSE {r0:.3f}")
bu = ent.groupby("userID").rating.mean() - mu      # sesgo usuario
bi = ent.groupby("placeID").rating.mean() - mu     # sesgo restaurante
for nombre, p in [
        ("media del usuario", mu + pru.userID.map(bu).fillna(0)),
        ("media del restaurante", mu + pru.placeID.map(bi).fillna(0)),
        ("usuario + restaurante", mu + pru.userID.map(bu).fillna(0)
         + pru.placeID.map(bi).fillna(0))]:
    print(f"{nombre:<22} RMSE {rmse(np.clip(p, 0, 2)):.3f}")
nuevos = (~pru.userID.isin(ent.userID)).sum()
print("valoraciones de prueba de usuarios sin historial:", nuevos)

`k3_vecinos.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

r = pd.read_csv("rating_final.csv")
ent, pru = train_test_split(r, test_size=0.2, random_state=0)
mu = ent.rating.mean()
bu = ent.groupby("userID").rating.mean() - mu
bi = (ent.rating - mu - ent.userID.map(bu)).groupby(
    ent.placeID).mean()                       # sesgo del restaurante
base = lambda u, i: mu + bu.get(u, 0) + bi.get(i, 0)
ent = ent.assign(res=ent.rating - [base(u, i) for u, i in
                                    zip(ent.userID, ent.placeID)])
R = ent.pivot(index="userID", columns="placeID", values="res")
A, M = R.fillna(0).to_numpy(), R.notna().to_numpy().astype(float)
comunes = M.T @ M                             # usuarios en común
norma = np.sqrt((A ** 2).sum(0)) + 1e-9
cos = A.T @ A / np.outer(norma, norma)
def predecir(u, i, k, encoger):
    S = cos * comunes / (comunes + encoger + 1e-9)  # pocos: menos
    if i not in R.columns:
        return base(u, i)
    vistos = R.loc[u].dropna()
    s = pd.Series(S[R.columns.get_loc(i)], R.columns)[vistos.index]
    s = s[s > 0].nlargest(k)
    if s.empty:
        return base(u, i)
    return base(u, i) + (s * vistos[s.index]).sum() / s.abs().sum()
for k, encoger in [(5, 0), (5, 5), (10, 5), (10, 20)]:
    p = np.clip([predecir(u, i, k, encoger) for u, i in
                 zip(pru.userID, pru.placeID)], 0, 2)
    print(f"{k:2d} vecinos, encogimiento {encoger:2d}: RMSE "
          f"{np.sqrt(np.mean((pru.rating - p) ** 2)):.3f}")
p0 = np.clip([base(u, i) for u, i in zip(pru.userID, pru.placeID)],
             0, 2)
print(f"solo sesgos (sin vecinos):         RMSE "
      f"{np.sqrt(np.mean((pru.rating - p0) ** 2)):.3f}")

`k4_factorizar.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

r = pd.read_csv("rating_final.csv")
ent, pru = train_test_split(r, test_size=0.2, random_state=0)
uid = {u: n for n, u in enumerate(r.userID.unique())}
iid = {i: n for n, i in enumerate(r.placeID.unique())}
def indices(d):
    return d.userID.map(uid).to_numpy(), d.placeID.map(iid).to_numpy()
ue, ie = indices(ent); up, ip = indices(pru)
mu = ent.rating.mean()
def entrenar(k, lam, epocas=60, tasa=0.02, semilla=0):
    rng = np.random.default_rng(semilla)
    P = rng.normal(0, 0.1, (len(uid), k))     # factores de usuario
    Q = rng.normal(0, 0.1, (len(iid), k))  # factores restaurante
    bu, bi = np.zeros(len(uid)), np.zeros(len(iid))
    for _ in range(epocas):
        for n in rng.permutation(len(ue)):
            u, i = ue[n], ie[n]
            e = ent.rating.iat[n] - (mu + bu[u] + bi[i] + P[u] @ Q[i])
            bu[u] += tasa * (e - lam * bu[u])
            bi[i] += tasa * (e - lam * bi[i])
            P[u], Q[i] = (P[u] + tasa * (e * Q[i] - lam * P[u]),
                          Q[i] + tasa * (e * P[u] - lam * Q[i]))
    return lambda u, i: mu + bu[u] + bi[i] + (P[u] * Q[i]).sum(1)
for k, lam in [(2, 0.1), (5, 0.1), (5, 0.01), (20, 0.1)]:
    f = entrenar(k, lam)
    p = np.clip(f(up, ip), 0, 2)
    print(f"k={k:2d}, lambda={lam}: RMSE "
          f"{np.sqrt(np.mean((pru.rating - p) ** 2)):.3f}")

`k5_contenido.py`

In [ ]:
import pandas as pd

r = pd.read_csv("rating_final.csv")
cu = pd.read_csv("chefmozcuisine.csv")       # cocinas de cada local
uc = pd.read_csv("usercuisine.csv")          # cocinas que prefiere
g = pd.read_csv("geoplaces2.csv", usecols=["placeID", "name"])
coc = cu.groupby("placeID").Rcuisine.agg(set)
pref = uc.groupby("userID").Rcuisine.agg(set)
r["coincide"] = [bool(pref.get(u, set()) & coc.get(i, set()))
                 for u, i in zip(r.userID, r.placeID)]
print("valoraciones con cocina conocida del local:",
      r.placeID.isin(coc.index).sum(), "de", len(r))
print(r.groupby("coincide").rating.agg(["mean", "size"]).round(3)
      .to_string())
def recomendar(cocinas, n=3):               # usuario sin historial
    media = r.groupby("placeID").rating.agg(["mean", "size"])
    ok = [i for i in media.index if coc.get(i, set()) & cocinas]
    m = media.loc[ok].query("size >= 5")
    return m.sort_values("mean", ascending=False).head(n).join(
        g.set_index("placeID").name)
print(recomendar({"Mexican"}).round(2).to_string())

## 3.6 · Aprendizaje por refuerzo

`l1_bandido.py`

In [ ]:
import numpy as np

def jugar(estrategia, p, pasos=5000, semilla=0):
    rng = np.random.default_rng(semilla)
    n, q, total = np.zeros(len(p)), np.zeros(len(p)), 0
    for t in range(1, pasos + 1):
        if estrategia == "voraz":
            a = q.argmax() if n.all() else n.argmin()
        elif estrategia == "e-voraz":
            a = rng.integers(len(p)) if rng.random() < 0.1 \
                else q.argmax()
        else:                                # UCB1: optimismo
            a = n.argmin() if not n.all() else \
                (q + np.sqrt(2 * np.log(t) / n)).argmax()
        r = rng.random() < p[a]
        n[a] += 1; q[a] += (r - q[a]) / n[a]; total += r
    return pasos * max(p) - total            # arrepentimiento

p = [0.04, 0.05, 0.06, 0.08, 0.10]  # clics de 5 anuncios
for e in ["voraz", "e-voraz", "UCB1"]:
    perd = [jugar(e, p, semilla=s) for s in range(50)]
    print(f"{e:<8} clics perdidos frente al mejor anuncio: "
          f"media {np.mean(perd):6.1f}, peor {np.max(perd):6.1f}")
print(f"si se elige al azar: {5000 * (max(p) - np.mean(p)):.1f}")

`l2_taxi.py`

In [ ]:
import numpy as np
import gymnasium as gym

env = gym.make("Taxi-v4")
nS, nA = env.observation_space.n, env.action_space.n
print("estados:", nS, "| acciones:", nA)
Q = np.zeros((nS, nA))
rng = np.random.default_rng(0)
alfa, gamma = 0.1, 0.99
medias = []
for ep in range(1, 5001):
    s, _ = env.reset(seed=int(rng.integers(1_000_000)))
    eps = max(0.01, 1 - ep / 2000)  # explora menos
    total, fin = 0, False
    while not fin:
        a = rng.integers(nA) if rng.random() < eps else Q[s].argmax()
        s2, r, term, trunc, _ = env.step(a)
        objetivo = r + gamma * Q[s2].max() * (not term)
        Q[s, a] += alfa * (objetivo - Q[s, a])   # Q-learning
        s, total, fin = s2, total + r, term or trunc
    medias.append(total)
    if ep in (100, 1000, 2000, 5000):
        print(f"episodios {ep:5d}: recompensa media de los últimos"
              f" 100 = {np.mean(medias[-100:]):7.1f}")
ok, pasos = 0, []
for semilla in range(100):                   # evaluar sin explorar
    s, _ = env.reset(seed=10_000 + semilla)
    for t in range(200):
        s, r, term, trunc, _ = env.step(Q[s].argmax())
        if term:
            ok += 1; pasos.append(t + 1); break
print(f"viajes completados: {ok} de 100, pasos medios"
      f" {np.mean(pasos):.1f}")

`l3_valor.py`

In [ ]:
import numpy as np
import gymnasium as gym

env = gym.make("Taxi-v4")
P = env.unwrapped.P   # modelo: P[s][a] = [(prob, s2, r, fin)]
nS, nA, gamma = 500, 6, 0.99
V = np.zeros(nS)
for it in range(1, 1000):    # iteración de valor (Bellman)
    Q = np.array([[sum(p * (r + gamma * V[s2] * (not f))
                       for p, s2, r, f in P[s][a])
                   for a in range(nA)] for s in range(nS)])
    cambio = np.abs(Q.max(1) - V).max()
    V = Q.max(1)
    if cambio < 1e-6:
        break
print("iteraciones hasta converger:", it)
ok, pasos = 0, []
for semilla in range(100):   # mismos viajes que con Q-learning
    s, _ = env.reset(seed=10_000 + semilla)
    for t in range(200):
        s, r, term, trunc, _ = env.step(Q[s].argmax())
        if term:
            ok += 1; pasos.append(t + 1); break
print(f"viajes completados: {ok} de 100, pasos medios"
      f" {np.mean(pasos):.1f}")

**Prueba.** En `l1_bandido.py`, cambia el 10 % de exploración de la estrategia ε-voraz por 5 % y 20 %. ¿Cuál pierde menos clics?